In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import math


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- split_empty_df ---
def FIX_SPLIT_EMPTY_DF_TRAIN_TEST_SPLIT(df, test_size=0.2, random_state=42):
    n_test = math.ceil(len(df) * test_size) if isinstance(test_size, float) else int(test_size)
    n_test = max(0, min(n_test, len(df)))
    return df.iloc[:-n_test or None].reset_index(drop=True), df.iloc[-n_test:].reset_index(drop=True)

class Dataset:
    def __init__(self, df=None, x_columns=None, y_columns=None, w_columns=None):
        self.df = df if df is not None else pd.DataFrame({"x": [1, 2, 3, 4], "y": [0, 1, 0, 1], "w": [1, 0, 1, 0]})
        self.x_columns = x_columns or ["x"]
        self.y_columns = y_columns or ["y"]
        self.w_columns = w_columns or ["w"]
    def to_pandas(self):
        return self.df.to_pandas() if isinstance(self.df, pl.DataFrame) else self.df.copy()
    def to_polars(self):
        return self.df if isinstance(self.df, pl.DataFrame) else pl.from_pandas(self.df)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_split_empty_df(train_test_split):
    def split(
        ds: Dataset,
        test_frac: float | None = None,
        test_n: float | None = None,
        random_state: int = 42,
    ) -> tuple[Dataset, Dataset]:
        if test_frac == 0 or test_n == 0:
            return ds, Dataset(
                pd.DataFrame(columns=ds.to_pandas().columns),
                ds.x_columns,
                ds.y_columns,
                ds.w_columns,
            )
        if test_frac == 1 or test_n == 1:
            return Dataset(
                pd.DataFrame(columns=ds.to_pandas().columns),
                ds.x_columns,
                ds.y_columns,
                ds.w_columns,
            ), ds

        test_size = test_frac if test_frac is not None else test_n
        train_df, test_df = train_test_split(
            ds.to_pandas(), test_size=test_size, random_state=random_state
        )
        return (
            Dataset(train_df, ds.x_columns, ds.y_columns, ds.w_columns),
            Dataset(test_df, ds.x_columns, ds.y_columns, ds.w_columns),
        )
    return split

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_split_empty_df(train_test_split):

    def split(
        ds: Dataset,
        test_frac: float | None = None,
        test_n: float | None = None,
        random_state: int = 42,
    ) -> tuple[Dataset, Dataset]:
        if test_frac == 0 or test_n == 0:
            return ds, Dataset(
                pl.DataFrame(schema={col: pl.Null for col in ds.to_pandas().columns}),
                ds.x_columns,
                ds.y_columns,
                ds.w_columns,
            )
        if test_frac == 1 or test_n == 1:
            return Dataset(
                pl.DataFrame(schema={col: pl.Null for col in ds.to_pandas().columns}),
                ds.x_columns,
                ds.y_columns,
                ds.w_columns,
            ), ds

        test_size = test_frac if test_frac is not None else test_n
        train_df, test_df = train_test_split(
            ds.to_pandas(), test_size=test_size, random_state=random_state
        )
        return (
            Dataset(train_df, ds.x_columns, ds.y_columns, ds.w_columns),
            Dataset(test_df, ds.x_columns, ds.y_columns, ds.w_columns),
        )
    return split

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: split_empty_df ===

def _split_fixture(kind):
    data = {"x": [1, 2, 3, 4], "y": [0, 1, 0, 1], "w": [1, 0, 1, 0]}
    frame = pd.DataFrame(data) if kind == "pandas" else pl.DataFrame(data)
    return Dataset(frame, ["x"], ["y"], ["w"])

def _check_split_contract(before_pair, generated_pair, label):
    _before_train, _before_test = before_pair
    _gen_train, _gen_test = generated_pair
    _before_shapes = (_before_train.to_pandas().shape, _before_test.to_pandas().shape)
    _gen_shapes = (_gen_train.to_polars().shape, _gen_test.to_polars().shape)
    assert _before_shapes == _gen_shapes, (
        f"partition shapes differ: before={_before_shapes}, gen={_gen_shapes}"
    )
    assert (
        _before_train.x_columns,
        _before_train.y_columns,
        _before_train.w_columns,
    ) == (
        _gen_train.x_columns,
        _gen_train.y_columns,
        _gen_train.w_columns,
    )
    _before_all = pd.concat(
        [_before_train.to_pandas(), _before_test.to_pandas()], ignore_index=True
    )
    # Empty pandas frames created with ``columns=...`` do not retain the source
    # dtypes. Polars represents that contract as Null columns, so concatenate
    # with coercion before comparing row preservation.
    _gen_all = pl.concat(
        [_gen_train.to_polars(), _gen_test.to_polars()], how="vertical_relaxed"
    )
    compare(_before_all, _gen_all, label)

# L1 executes the returned generated function, not only its definition.
try:
    _generated_split = gen_split_empty_df(FIX_SPLIT_EMPTY_DF_TRAIN_TEST_SPLIT)
    _r = _generated_split(_split_fixture("polars"), test_frac=0.5, random_state=7)
    assert len(_r) == 2
    print("✅ L1 smoke gen_split_empty_df: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_split_empty_df: {type(_e).__name__}: {_e}")

try:
    _before_split = before_split_empty_df(FIX_SPLIT_EMPTY_DF_TRAIN_TEST_SPLIT)
    _rb = _before_split(_split_fixture("pandas"), test_frac=0.5, random_state=7)
    assert len(_rb) == 2
    print("✅ L1 smoke before_split_empty_df: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_split_empty_df: {type(_e).__name__}: {_e}")

# L2 compares partition sizes, metadata, and preservation of all input rows.
# Exact row membership is intentionally unordered because the two libraries use
# different seeded shuffle implementations.
try:
    _before_split = before_split_empty_df(FIX_SPLIT_EMPTY_DF_TRAIN_TEST_SPLIT)
    _generated_split = gen_split_empty_df(FIX_SPLIT_EMPTY_DF_TRAIN_TEST_SPLIT)
    _check_split_contract(
        _before_split(_split_fixture("pandas"), test_frac=0.5, random_state=7),
        _generated_split(_split_fixture("polars"), test_frac=0.5, random_state=7),
        "split_empty_df",
    )
    print("✅ L2 equivalence split_empty_df contract: MATCH")
except Exception as _e:
    print(f"❌ L2 equivalence split_empty_df: setup error — {type(_e).__name__}: {_e}")

# L3 exercises both early returns plus fractional and integer split branches.
try:
    _before_split = before_split_empty_df(FIX_SPLIT_EMPTY_DF_TRAIN_TEST_SPLIT)
    _generated_split = gen_split_empty_df(FIX_SPLIT_EMPTY_DF_TRAIN_TEST_SPLIT)
    for _case, _kwargs in [
        ("zero", {"test_frac": 0}),
        ("one", {"test_frac": 1}),
        ("fraction", {"test_frac": 0.5}),
        ("integer", {"test_n": 2}),
    ]:
        _check_split_contract(
            _before_split(_split_fixture("pandas"), **_kwargs),
            _generated_split(_split_fixture("polars"), **_kwargs),
            f"L3 branch split_empty_df {_case}",
        )
    print("✅ L3 branch split_empty_df all paths: MATCH")
except Exception as _e:
    print(f"❌ L3 branch split_empty_df: {type(_e).__name__}: {_e}")
